# Advanced Rotating Equipment Fault Diagnosis

From the [Sisyphean Gridworks ML Playground](https://sgridworks.com/ml-playground/guides/19-advanced-rotating-equipment-diagnostics.html)

## Setup

Clone the repository and install dependencies. Run this cell first.

In [ ]:
import os, subprocess

# Colab: clone the repo and cd into it
# Local: detect if we are already inside the repo
if not os.path.exists('sisyphean-power-and-light'):
    if os.path.exists('../sisyphean-power-and-light'):
        os.chdir('..')  # running from notebooks/ subfolder
    else:
        subprocess.run(['git', 'clone', 'https://github.com/SGridworks/Dynamic-Network-Model.git'], capture_output=True)
        os.chdir('Dynamic-Network-Model')

print(f'Working directory: {os.getcwd()}')
# !pip install -q pandas numpy matplotlib seaborn scikit-learn pyarrow shap


## Step 0: Load Data and Filter to Running Periods

Guide 17 detected anomalies. Guide 18 built load-correlation models. This guide goes further: given that something is wrong, **what kind of fault is it?** We use multi-sensor fusion and supervised classification to distinguish between seal degradation, bearing wear, and misalignment.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler

# Load BFP hourly data
DATA_PATH = "sisyphean-power-and-light/generation/timeseries/bfp_train_hourly.parquet"
df = pd.read_parquet(DATA_PATH)
df["timestamp"] = pd.to_datetime(df["timestamp"])
df = df.set_index("timestamp").sort_index()

# Load design parameters for reference
with open("sisyphean-power-and-light/generation/reference/design_parameters.json") as f:
    design = json.load(f)

# Filter to running periods only (either pump)
mask_a = df["U1_BFPA_RUN_STATUS"] > 0
mask_b = df["U1_BFPB_RUN_STATUS"] > 0
mask_running = mask_a | mask_b
df_run = df.loc[mask_running].copy()

print(f"Total rows: {len(df):,}")
print(f"Running rows: {len(df_run):,}")
print(f"BFP-A running: {mask_a.sum():,}")
print(f"BFP-B running: {mask_b.sum():,}")
assert len(df) == 8784, f"Expected 8,784 rows, got {len(df)}"
print("\nSetup verified.")

## Step 1: Feature Engineering -- Derived Diagnostic Features

Raw sensor values are useful, but domain-engineered features are far more discriminating. We compute:
- **Pump efficiency proxy**: flow * differential pressure / power (higher = more efficient)
- **Vibration DE/NDE ratio**: imbalance between drive-end and non-drive-end (>1 suggests DE-side issue)
- **Seal leak rate of change**: rising rate indicates progressive seal failure
- **Bearing temp rate of change**: sudden rise indicates lubrication or clearance problems

In [ ]:
# We build features for whichever pump is running at each timestamp
# First, create unified columns for the active pump's sensors

df_run["active_pump"] = "A"
df_run.loc[df_run["U1_BFPB_RUN_STATUS"] > 0, "active_pump"] = "B"

def get_active(row, tag_suffix):
    """Return the value from the active pump's tag."""
    prefix = "U1_BFPA_" if row["active_pump"] == "A" else "U1_BFPB_"
    return row[prefix + tag_suffix]

# Map active pump columns
tag_suffixes = [
    "FW_FLOW", "DISCH_PRESS", "MTR_POWER", "MTR_CURRENT", "SPEED",
    "BRG_DE_TEMP", "BRG_NDE_TEMP", "THR_ACT_TEMP",
    "VIB_DE_X", "VIB_NDE_X", "SEAL_DE_LEAK", "AXIAL_DISP",
    "LO_HDR_PRESS",
]

for suffix in tag_suffixes:
    df_run[suffix] = np.where(
        df_run["active_pump"] == "A",
        df_run[f"U1_BFPA_{suffix}"],
        df_run[f"U1_BFPB_{suffix}"]
    )

# Derived features
# 1. Efficiency proxy (flow * dp / power) -- avoid division by zero
df_run["efficiency_proxy"] = (
    df_run["FW_FLOW"] * df_run["DISCH_PRESS"] /
    df_run["MTR_POWER"].clip(lower=1)
)

# 2. Vibration DE/NDE ratio
df_run["vib_de_nde_ratio"] = (
    df_run["VIB_DE_X"] / df_run["VIB_NDE_X"].clip(lower=0.1)
)

# 3. Seal leak rate of change (hourly delta)
df_run["seal_leak_roc"] = df_run["SEAL_DE_LEAK"].diff().fillna(0)

# 4. Bearing temp rates of change
df_run["brg_de_roc"] = df_run["BRG_DE_TEMP"].diff().fillna(0)
df_run["brg_nde_roc"] = df_run["BRG_NDE_TEMP"].diff().fillna(0)

# 5. Bearing DE-NDE temperature spread
df_run["brg_temp_spread"] = df_run["BRG_DE_TEMP"] - df_run["BRG_NDE_TEMP"]

# 6. Power per unit flow (specific energy)
df_run["specific_power"] = (
    df_run["MTR_POWER"] / df_run["FW_FLOW"].clip(lower=1)
)

print("Derived features computed:")
derived = ["efficiency_proxy", "vib_de_nde_ratio", "seal_leak_roc",
           "brg_de_roc", "brg_nde_roc", "brg_temp_spread", "specific_power"]
print(df_run[derived].describe().round(3).to_string())

## Step 2: Label Windows with Known Fault Types

The dataset contains three embedded fault scenarios documented in `design_parameters.json`. We assign labels to each hour based on the known fault windows. This creates a supervised classification problem with four classes.

In [ ]:
# Label fault windows from design_parameters.json
faults = design["fault_scenarios_embedded"]
print("Known fault scenarios:")
for f in faults:
    print(f"  {f['id']}: {f['name']} ({f['start'][:10]} to {f['end'][:10]})")

# Assign labels: 0=healthy, 1=seal, 2=bearing, 3=misalignment
df_run["fault_label"] = 0  # default: healthy
df_run["fault_name"] = "healthy"

# Seal degradation: BFP-A, Apr 1 - Jun 1
seal_mask = (
    (df_run.index >= "2024-04-01") &
    (df_run.index < "2024-06-01") &
    (df_run["active_pump"] == "A")
)
df_run.loc[seal_mask, "fault_label"] = 1
df_run.loc[seal_mask, "fault_name"] = "seal"

# Bearing wear: BFP-B, Jul 15 - Aug 20
brg_mask = (
    (df_run.index >= "2024-07-15") &
    (df_run.index < "2024-08-21") &
    (df_run["active_pump"] == "B")
)
df_run.loc[brg_mask, "fault_label"] = 2
df_run.loc[brg_mask, "fault_name"] = "bearing"

# Misalignment: BFP-A, Oct 15 - Dec 31
align_mask = (
    (df_run.index >= "2024-10-15") &
    (df_run.index <= "2024-12-31") &
    (df_run["active_pump"] == "A")
)
df_run.loc[align_mask, "fault_label"] = 3
df_run.loc[align_mask, "fault_name"] = "misalignment"

print("\nLabel distribution:")
print(df_run["fault_name"].value_counts().to_string())

## Step 3: Train Multi-Class Random Forest

We use all raw sensor features plus the derived features to train a Random Forest classifier. The model must learn to distinguish the four classes from the joint pattern of all sensors.

In [ ]:
# Define feature set: raw sensors + derived features
raw_features = [
    "BRG_DE_TEMP", "BRG_NDE_TEMP", "THR_ACT_TEMP",
    "VIB_DE_X", "VIB_NDE_X", "SEAL_DE_LEAK", "AXIAL_DISP",
    "LO_HDR_PRESS", "MTR_POWER", "MTR_CURRENT", "SPEED",
    "FW_FLOW", "DISCH_PRESS",
]
derived_features = [
    "efficiency_proxy", "vib_de_nde_ratio", "seal_leak_roc",
    "brg_de_roc", "brg_nde_roc", "brg_temp_spread", "specific_power",
]
all_features = raw_features + derived_features

# Also include unit load as context
all_features.append("U1_UNIT_MW_GROSS")

X = df_run[all_features].fillna(0)
y = df_run["fault_label"]

# Stratified train/test split (70/30)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42, stratify=y
)

print(f"Training samples: {len(X_train):,}")
print(f"Test samples:     {len(X_test):,}")
print(f"Features:         {len(all_features)}")

# Train Random Forest
rf = RandomForestClassifier(
    n_estimators=300,
    max_depth=15,
    min_samples_leaf=5,
    class_weight="balanced",  # handle class imbalance
    random_state=42,
    n_jobs=-1,
)
rf.fit(X_train, y_train)

print("\nRandom Forest trained.")
print(f"Train accuracy: {rf.score(X_train, y_train):.4f}")
print(f"Test accuracy:  {rf.score(X_test, y_test):.4f}")

## Step 4: Confusion Matrix and Classification Report

The confusion matrix shows where the model confuses one fault type for another. In rotating equipment diagnostics, some confusion between related faults is expected (e.g., severe misalignment can cause secondary bearing temperature rise).

In [ ]:
# Predictions
y_pred = rf.predict(X_test)
class_names = ["healthy", "seal", "bearing", "misalignment"]

# Classification report
print("Classification Report:")
print(classification_report(y_test, y_pred, target_names=class_names))

# Confusion matrix
cm = confusion_matrix(y_test, y_pred)
fig, ax = plt.subplots(figsize=(8, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm, display_labels=class_names)
disp.plot(ax=ax, cmap="Blues", values_format="d")
ax.set_title("Confusion Matrix: Multi-Class Fault Diagnosis")
plt.tight_layout()
plt.show()

## Step 5: SHAP Analysis -- Which Sensors Matter Most for Each Fault?

SHAP (SHapley Additive exPlanations) values quantify each feature's contribution to each prediction. For multi-class problems, SHAP reveals distinct fault signatures: seal faults should be driven by seal leakage features, bearing faults by temperature features, and misalignment by vibration features.

In [ ]:
import shap

# Use a sample for SHAP computation (full dataset is slow)
np.random.seed(42)
sample_idx = np.random.choice(len(X_test), size=min(500, len(X_test)), replace=False)
X_sample = X_test.iloc[sample_idx]

# Compute SHAP values using TreeExplainer (fast for tree-based models)
explainer = shap.TreeExplainer(rf)
shap_values_raw = explainer.shap_values(X_sample)

# Normalize SHAP output to a list of 2D arrays (one per class).
# Different SHAP versions return different formats:
#   - List of 2D arrays: [array(n_samples, n_features)] * n_classes  (older)
#   - 3D ndarray: (n_samples, n_features, n_classes)                 (newer)
#   - shap.Explanation object with .values property                   (newest)
n_classes = rf.n_classes_
n_samp = len(X_sample)
n_feat = len(all_features)

if hasattr(shap_values_raw, "values"):
    # shap.Explanation object
    sv = np.array(shap_values_raw.values)
    if sv.ndim == 3:
        shap_values = [sv[:, :, i] for i in range(sv.shape[2])]
    else:
        shap_values = [sv]
elif isinstance(shap_values_raw, np.ndarray) and shap_values_raw.ndim == 3:
    # 3D array -- detect axis order by matching known dimensions
    if shap_values_raw.shape == (n_classes, n_samp, n_feat):
        shap_values = [shap_values_raw[i] for i in range(n_classes)]
    else:
        # (n_samples, n_features, n_classes) -- most common in newer SHAP
        shap_values = [shap_values_raw[:, :, i] for i in range(shap_values_raw.shape[2])]
elif isinstance(shap_values_raw, list):
    shap_values = shap_values_raw
else:
    shap_values = [np.array(shap_values_raw)]

feature_names = all_features
print(f"SHAP values computed for {n_samp} samples.")
print(f"Number of classes: {len(shap_values)}")
print(f"Shape per class: {shap_values[0].shape}")
assert shap_values[0].shape == (n_samp, n_feat), \
    f"SHAP shape {shap_values[0].shape} != expected ({n_samp}, {n_feat})"

# Global feature importance by class (mean absolute SHAP)
fig, axes = plt.subplots(2, 2, figsize=(10, 5))
n_top = min(10, len(feature_names))

for idx, (ax, class_name) in enumerate(zip(axes.ravel(), class_names)):
    importance = np.abs(shap_values[idx]).mean(axis=0)
    top_k = np.argsort(importance)[-n_top:]
    ax.barh(range(len(top_k)), importance[top_k], color="#5FCCDB")
    ax.set_yticks(range(len(top_k)))
    ax.set_yticklabels([feature_names[i] for i in top_k], fontsize=7)
    ax.set_title(f"{class_name}", fontsize=10)
    ax.set_xlabel("Mean |SHAP|", fontsize=8)

plt.suptitle(f"Top {n_top} Features by Fault Type (SHAP Importance)", fontsize=12)
plt.tight_layout()
plt.show()

In [ ]:
# SHAP summary plot for the seal fault class (class 1)
# Uses the normalized shap_values list from the previous cell
print("SHAP Summary: Seal Fault Class")
print("Red = high feature value, Blue = low feature value")
print("Positive SHAP = pushes toward seal fault classification\n")

# Convert X_sample to numpy for compatibility with all SHAP versions
shap.summary_plot(shap_values[1], X_sample.values, feature_names=feature_names,
                  max_display=15, show=True)

## Step 6: Sliding Window Classifier for Real-Time Detection

In practice, a classifier runs continuously on a sliding window of recent data. We simulate this by scoring every hour in the dataset and tracking the classification confidence (probability) for each fault class over time.

In [ ]:
# Score every running hour and extract class probabilities
X_all = df_run[all_features].fillna(0)
probs = rf.predict_proba(X_all)
predictions = rf.predict(X_all)

# Add probabilities back to the dataframe
for i, name in enumerate(class_names):
    df_run[f"prob_{name}"] = probs[:, i]
df_run["predicted_fault"] = predictions
df_run["predicted_name"] = df_run["predicted_fault"].map(
    {i: n for i, n in enumerate(class_names)}
)

# Compute sliding window smoothed probabilities (24h window)
for name in class_names:
    df_run[f"prob_{name}_smooth"] = (
        df_run[f"prob_{name}"].rolling(24, min_periods=1).mean()
    )

print("Classification probabilities computed for all running hours.")
print(f"\nPredicted label distribution:")
print(df_run["predicted_name"].value_counts().to_string())

## Step 7: Plot Classification Confidence Over Time

This is the key diagnostic output: how the model's confidence in each fault class evolves over the year. You should see the seal probability rising in April, bearing probability rising in late July, and misalignment probability rising in October.

In [ ]:
# Stacked area plot of classification confidence over time
fig, axes = plt.subplots(2, 1, figsize=(10, 5), sharex=True)

colors = {"healthy": "#5FCCDB", "seal": "#D69E2E",
          "bearing": "#E53E3E", "misalignment": "#718096"}

# Top: smoothed probabilities
for name in class_names:
    axes[0].plot(df_run.index, df_run[f"prob_{name}_smooth"],
                 color=colors[name], linewidth=1.2, label=name)
axes[0].set_ylabel("Smoothed Probability")
axes[0].set_title("Fault Classification Confidence Over Time (24h smoothed)")
axes[0].legend(fontsize=8, ncol=4, loc="upper right")
axes[0].set_ylim(0, 1)

# Shade known fault periods
for ax in axes:
    ax.axvspan(pd.Timestamp("2024-04-01"), pd.Timestamp("2024-06-01"),
               alpha=0.08, color="#D69E2E")
    ax.axvspan(pd.Timestamp("2024-07-15"), pd.Timestamp("2024-08-21"),
               alpha=0.08, color="#E53E3E")
    ax.axvspan(pd.Timestamp("2024-10-15"), pd.Timestamp("2024-12-31"),
               alpha=0.08, color="#718096")

# Bottom: predicted class as colored scatter
label_colors = df_run["predicted_fault"].map(
    {0: "#5FCCDB", 1: "#D69E2E", 2: "#E53E3E", 3: "#718096"}
)
axes[1].scatter(df_run.index, df_run["predicted_fault"], c=label_colors,
                s=2, alpha=0.5)
axes[1].set_yticks([0, 1, 2, 3])
axes[1].set_yticklabels(class_names)
axes[1].set_ylabel("Predicted Class")
axes[1].set_xlabel("Date")
axes[1].set_title("Predicted Fault Class (per hour)")

plt.tight_layout()
plt.show()

## Step 8: Fault Signatures Summary

Each fault type has a distinct multi-sensor signature. This table summarizes the key differentiators learned by the model and confirmed by SHAP analysis.

In [ ]:
# Compute average feature values per fault class (normalized to healthy baseline)
healthy_means = df_run.loc[df_run["fault_name"] == "healthy", all_features].mean()
healthy_stds = df_run.loc[df_run["fault_name"] == "healthy", all_features].std()

# Z-score relative to healthy baseline
signatures = {}
for fault in ["seal", "bearing", "misalignment"]:
    fault_means = df_run.loc[df_run["fault_name"] == fault, all_features].mean()
    z_scores = (fault_means - healthy_means) / healthy_stds.clip(lower=0.001)
    signatures[fault] = z_scores

sig_df = pd.DataFrame(signatures)
sig_df.index.name = "Feature"

# Show features with |z| > 1 for any fault
significant = sig_df[(sig_df.abs() > 1).any(axis=1)]

print("Fault Signature Table (z-scores relative to healthy baseline)")
print("Values > 1 = elevated, < -1 = depressed compared to healthy operation")
print("=" * 70)
print(significant.round(2).to_string())

# Visualize as heatmap
fig, ax = plt.subplots(figsize=(8, 5))
sns.heatmap(significant.T, annot=True, fmt=".1f", cmap="RdBu_r", center=0,
            linewidths=0.5, ax=ax, vmin=-5, vmax=5)
ax.set_title("Fault Signatures: Z-Score Deviation from Healthy Baseline")
ax.set_ylabel("Fault Type")
plt.tight_layout()
plt.show()

## What You Built and Next Steps

In this guide you:

1. **Engineered** 7 derived diagnostic features from raw sensor data (efficiency proxy, vibration ratios, rates of change, temperature spreads, specific power)
2. **Labeled** running hours with 4 fault classes using documented fault windows from the design parameters
3. **Trained** a 300-tree Random Forest classifier achieving multi-class discrimination between healthy, seal, bearing, and misalignment conditions
4. **Evaluated** with confusion matrix and per-class precision/recall/F1 scores
5. **Explained** model decisions using SHAP, revealing distinct sensor signatures for each fault type
6. **Simulated** real-time classification by scoring every running hour and plotting probability trajectories
7. **Summarized** fault signatures as z-score deviations from healthy baseline

The key insight: each fault type has a unique multi-sensor fingerprint. Seal faults show up primarily in leakage rate and efficiency degradation. Bearing faults manifest as temperature rise at the affected end. Misalignment produces elevated vibration at both ends with characteristic DE/NDE ratio changes.

**Next steps:**
- **Guide 20**: Build a physics-informed digital twin using OEM pump curves to track performance against design intent